# 🛒 ShopEase — FBDA 2026 Dynamic Analytical Dashboard

**Project:** Dynamic Analytical Dashboard with Python  
**Student:** Khushi Nagpal  
**Enrollment:** 075112  
**Group ID:** `112_096_074`  
**Fixed random seed:** `112096074`  
**Required sample:** `2,500` random records  
**Dataset:** [Kaggle — ShopEase Raw Orders Dataset](https://www.kaggle.com/datasets/mernahabib/shopease-raw-orders-dataset)

### Submission design
This notebook is written to follow the supplied FBDA project rubric. Streamlit is treated as the **hero deliverable**, while the notebook is the reproducible analytical engine.

The notebook:
1. installs the required libraries;
2. downloads the Kaggle data;
3. audits and cleans the source;
4. creates the fixed 2,500-record sample;
5. classifies variables and documents missing data;
6. performs all applicable descriptive, categorical, visual, inferential, correlation and regression analyses;
7. generates the Word analytical report;
8. writes the Streamlit application;
9. starts Streamlit in Colab;
10. creates a secure ngrok public dashboard URL.

**Credential safety:** the ngrok token is never hard-coded. Use a Colab Secret called `NGROK_AUTHTOKEN` or enter the token privately when prompted.

In [ ]:
# ============================================================
# CELL 1 — Install the project libraries
# ============================================================
!pip -q install pandas numpy matplotlib seaborn scipy statsmodels streamlit python-docx kagglehub pyngrok

print("All required FBDA libraries are installed.")

In [ ]:
# ============================================================
# CELL 2 — Imports, project configuration and reproducibility
# ============================================================
import os
import random
import math
import time
import subprocess
import getpass
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

GROUP_ID = "112_096_074"
SEED = 112096074
SAMPLE_SIZE = 2500
STUDENT_NAME = "Khushi Nagpal"
ENROLLMENT = "075112"
KAGGLE_DATASET = "mernahabib/shopease-raw-orders-dataset"
DATA_SOURCE = "https://www.kaggle.com/datasets/mernahabib/shopease-raw-orders-dataset"

random.seed(SEED)
np.random.seed(SEED)

Path("data").mkdir(exist_ok=True)
Path("report_assets").mkdir(exist_ok=True)

print("Group ID:", GROUP_ID)
print("Seed:", SEED)
print("Required sample size:", SAMPLE_SIZE)

In [ ]:
# ============================================================
# CELL 3 — Download the public Kaggle dataset
# ============================================================
import kagglehub

download_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("Kaggle dataset downloaded to:", download_path)

csv_files = list(download_path.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError("No CSV file was found inside the downloaded Kaggle dataset.")

# If the dataset contains more than one CSV, use the largest transactional CSV.
csv_path = max(csv_files, key=lambda p: p.stat().st_size)

print("Selected source CSV:", csv_path)
print("Source CSV size (KB):", round(csv_path.stat().st_size / 1024, 2))

In [ ]:
# ============================================================
# CELL 4 — Load raw data and perform a first audit
# ============================================================
raw_df = pd.read_csv(csv_path)

print("RAW DATA AUDIT")
print("-" * 60)
print("Rows:", f"{len(raw_df):,}")
print("Columns:", raw_df.shape[1])
print("Memory usage (MB):", round(raw_df.memory_usage(deep=True).sum() / 1024**2, 2))
print("Duplicate rows:", int(raw_df.duplicated().sum()))
print("\nColumn names:")
print(list(raw_df.columns))

display(raw_df.head())
display(raw_df.dtypes.to_frame("Pandas Data Type"))

In [ ]:
# ============================================================
# CELL 5 — Cleaning and data-type standardization
# ============================================================
def norm(x):
    return (
        str(x).strip().lower()
        .replace(" ", "_").replace("-", "_").replace("/", "_")
        .replace("(", "").replace(")", "").replace("%", "pct")
    )

clean_df = raw_df.copy()
clean_df.columns = [str(c).strip() for c in clean_df.columns]

# Convert obvious date/time fields only when at least 50% of values parse.
for c in clean_df.columns:
    if any(k in norm(c) for k in ["date", "datetime", "timestamp", "time"]):
        converted = pd.to_datetime(clean_df[c], errors="coerce")
        if converted.notna().mean() >= 0.50:
            clean_df[c] = converted

# Convert object columns to numeric only when at least 85% of values are numeric-looking.
for c in clean_df.columns:
    if clean_df[c].dtype == "object":
        cleaned = (
            clean_df[c].astype(str)
            .str.replace(",", "", regex=False)
            .str.replace(r"[$₹€£]", "", regex=True)
            .str.replace("%", "", regex=False)
            .str.strip()
        )
        numeric = pd.to_numeric(cleaned, errors="coerce")
        if numeric.notna().mean() >= 0.85:
            clean_df[c] = numeric

print("Cleaning completed.")
display(clean_df.dtypes.to_frame("Cleaned Data Type"))

In [ ]:
# ============================================================
# CELL 6 — Create the mandated fixed 2,500-record sample
# ============================================================
if len(clean_df) < SAMPLE_SIZE:
    raise ValueError(
        f"The source dataset contains only {len(clean_df):,} rows. "
        f"The project requires {SAMPLE_SIZE:,} records."
    )

random.seed(SEED)
np.random.seed(SEED)

sample_df = (
    clean_df
    .sample(n=SAMPLE_SIZE, random_state=SEED)
    .reset_index(drop=True)
)

sample_path = Path("data/shopease_sample_112_096_074.csv")
sample_df.to_csv(sample_path, index=False)

print("Fixed sample created.")
print("Sample shape:", sample_df.shape)
print("Sample file:", sample_path)
print("Sample file size (KB):", round(sample_path.stat().st_size / 1024, 2))
display(sample_df.head())

In [ ]:
# ============================================================
# CELL 7 — Semantic column detection
# ============================================================
def find_col(df, aliases):
    nmap = {norm(c): c for c in df.columns}
    for a in aliases:
        if norm(a) in nmap:
            return nmap[norm(a)]
    for a in aliases:
        na = norm(a)
        for n, original in nmap.items():
            if na in n or n in na:
                return original
    return None

date_col = find_col(sample_df, [
    "order_date","date","order_datetime","purchase_date",
    "transaction_date","created_at","order_time","datetime","timestamp"
])

amount_col = find_col(sample_df, [
    "sales","revenue","amount","total_amount","order_amount",
    "total","order_value","price","selling_price","gross_revenue","net_revenue"
])

qty_col = find_col(sample_df, [
    "quantity","qty","units","quantity_ordered","units_sold"
])

profit_col = find_col(sample_df, [
    "profit","net_profit","gross_profit"
])

num_cols = sample_df.select_dtypes(include=np.number).columns.tolist()
cat_cols = sample_df.select_dtypes(include=["object","category","bool"]).columns.tolist()
date_cols = [c for c in sample_df.columns if pd.api.types.is_datetime64_any_dtype(sample_df[c])]

print("Detected semantic fields")
print("Date:", date_col)
print("Amount / Sales:", amount_col)
print("Quantity:", qty_col)
print("Profit:", profit_col)
print("Numeric variables:", num_cols)
print("Categorical variables:", cat_cols)
print("Date variables:", date_cols)

In [ ]:
# ============================================================
# CELL 8 — Data description, variable classification and missing data
# ============================================================
def classify_variable(c, s):
    n = norm(c)
    if pd.api.types.is_datetime64_any_dtype(s):
        return "Date/Time", "Date/Time"
    if pd.api.types.is_numeric_dtype(s):
        if any(k in n for k in ["id", "code", "key"]):
            return "Non-Categorical", "Index (Nominal)"
        return "Non-Categorical", "Non-Index (Ratio)"
    if any(k in n for k in ["rank", "rating", "priority", "grade", "level"]):
        return "Categorical", "Ordinal"
    return "Categorical", "Nominal"

dictionary_rows = []
for c in sample_df.columns:
    broad, role = classify_variable(c, sample_df[c])
    dictionary_rows.append([
        c, str(sample_df[c].dtype), broad, role,
        int(sample_df[c].nunique(dropna=True)),
        int(sample_df[c].isna().sum())
    ])

variable_dictionary = pd.DataFrame(
    dictionary_rows,
    columns=["Variable","Pandas Type","Broad Type","Analytical Role","Unique Values","Missing"]
)

missing_table = pd.DataFrame({
    "Variable": sample_df.columns,
    "Missing Count": sample_df.isna().sum().values,
    "Missing %": (sample_df.isna().mean() * 100).values
}).sort_values("Missing %", ascending=False)

print("VARIABLE DICTIONARY")
display(variable_dictionary)

print("MISSING DATA AUDIT")
display(missing_table)

In [ ]:
# ============================================================
# CELL 9 — Descriptive statistics for every numeric variable
# ============================================================
if num_cols:
    descriptive = sample_df[num_cols].describe(
        percentiles=[.01, .05, .25, .50, .75, .95, .99]
    ).T
    descriptive["Range"] = descriptive["max"] - descriptive["min"]
    descriptive["Skewness"] = sample_df[num_cols].skew()
    descriptive["Kurtosis"] = sample_df[num_cols].kurtosis()

    central_tendency = pd.DataFrame({
        "Mean": sample_df[num_cols].mean(),
        "Median": sample_df[num_cols].median(),
        "Mode": [
            sample_df[c].mode(dropna=True).iloc[0]
            if not sample_df[c].mode(dropna=True).empty else np.nan
            for c in num_cols
        ]
    })

    print("DESCRIPTIVE STATISTICS")
    display(descriptive.round(4))

    print("CENTRAL TENDENCY")
    display(central_tendency.round(4))
else:
    descriptive = pd.DataFrame()
    central_tendency = pd.DataFrame()
    print("No numeric variables available.")

In [ ]:
# ============================================================
# CELL 10 — Categorical count, frequency and relative frequency
# ============================================================
frequency_tables = {}

for c in cat_cols:
    f = (
        sample_df[c].fillna("Missing").astype(str)
        .value_counts()
        .rename_axis(c)
        .reset_index(name="Count")
    )
    f["Relative Frequency"] = f["Count"] / f["Count"].sum()
    f["Relative Frequency (%)"] = f["Relative Frequency"] * 100
    frequency_tables[c] = f

for c, f in frequency_tables.items():
    print(f"\nCATEGORY: {c}")
    display(f.round(4))

In [ ]:
# ============================================================
# CELL 11 — Correlation analysis: Pearson and Spearman
# ============================================================
correlation_rows = []

if len(num_cols) >= 2:
    for i, a in enumerate(num_cols):
        for b in num_cols[i+1:]:
            z = sample_df[[a, b]].dropna()
            if len(z) >= 3:
                pear = stats.pearsonr(z[a], z[b])
                spear = stats.spearmanr(z[a], z[b])
                correlation_rows.append([
                    a, b, len(z),
                    pear.statistic, pear.pvalue,
                    spear.statistic, spear.pvalue
                ])

correlations = pd.DataFrame(
    correlation_rows,
    columns=["X","Y","N","Pearson_r","Pearson_p","Spearman_rho","Spearman_p"]
)

if not correlations.empty:
    correlations = correlations.sort_values(
        "Pearson_r", key=lambda s: s.abs(), ascending=False
    )
    display(correlations.round(4))
else:
    print("Correlation analysis requires at least two numeric variables.")

In [ ]:
# ============================================================
# CELL 12 — Required visualizations
# ============================================================
asset_dir = Path("report_assets")
asset_dir.mkdir(exist_ok=True)

primary_num = amount_col if amount_col in num_cols else (num_cols[0] if num_cols else None)
candidate_cats = [c for c in cat_cols if 2 <= sample_df[c].nunique(dropna=True) <= 10]
primary_cat = candidate_cats[0] if candidate_cats else (cat_cols[0] if cat_cols else None)

print("Primary numeric variable:", primary_num)
print("Primary categorical variable:", primary_cat)

# Line plot
if date_col and primary_num:
    t = sample_df[[date_col, primary_num]].dropna().copy()
    if not t.empty:
        t["Month"] = t[date_col].dt.to_period("M").astype(str)
        trend = t.groupby("Month")[primary_num].sum()
        fig, ax = plt.subplots(figsize=(10, 4.5))
        trend.plot(ax=ax, marker="o")
        ax.set_title(f"Line Plot — Monthly {primary_num}")
        ax.tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()
        plt.close(fig)

# Bar plot
if primary_cat and primary_num:
    bar = sample_df.groupby(primary_cat)[primary_num].sum().sort_values(ascending=False).head(10)
    fig, ax = plt.subplots(figsize=(9, 5))
    bar.sort_values().plot(kind="barh", ax=ax)
    ax.set_title(f"Bar Plot — Top {primary_cat} by {primary_num}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

# Histogram + box plot
if primary_num:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(sample_df[primary_num].dropna(), kde=True, ax=ax)
    ax.set_title(f"Histogram — {primary_num}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 3.5))
    sns.boxplot(x=sample_df[primary_num], ax=ax)
    ax.set_title(f"Box-Whisker Plot — {primary_num}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

# Violin
if primary_cat and primary_num and sample_df[primary_cat].nunique(dropna=True) <= 15:
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.violinplot(data=sample_df, x=primary_cat, y=primary_num, cut=0, inner="quartile", ax=ax)
    ax.tick_params(axis="x", rotation=45)
    ax.set_title(f"Violin Plot — {primary_num} by {primary_cat}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

# Heat map, pair plot, scatter
if len(num_cols) >= 2:
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(sample_df[num_cols].corr(), annot=True, fmt=".2f", center=0, ax=ax)
    ax.set_title("Pearson Correlation Heat Map")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    pair_cols = num_cols[:min(5, len(num_cols))]
    pair = sample_df[pair_cols].dropna()
    if len(pair) > 500:
        pair = pair.sample(500, random_state=SEED)
    if len(pair) >= 2:
        pg = sns.pairplot(pair)
        plt.show()
        plt.close(pg.fig)

    x, y = num_cols[:2]
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=sample_df, x=x, y=y, alpha=.65, ax=ax)
    ax.set_title(f"Scatter Plot — {x} vs {y}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

# Pie
if primary_cat:
    pie = sample_df[primary_cat].fillna("Missing").astype(str).value_counts().head(6)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.pie(pie.values, labels=pie.index, autopct="%1.1f%%")
    ax.set_title(f"Pie Plot — {primary_cat}")
    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# CELL 13 — Inferential statistics
# ============================================================
inferential = {}
ALPHA = 0.05

# Confidence interval + mean + normality
if primary_num:
    x = sample_df[primary_num].dropna()

    if len(x) >= 2:
        mean_x = x.mean()
        h = stats.t.ppf(.975, len(x)-1) * stats.sem(x)
        inferential["ci95"] = (mean_x - h, mean_x + h)
        inferential["one_sample_t_vs_0"] = stats.ttest_1samp(x, 0)

    if 3 <= len(x) <= 5000:
        inferential["shapiro"] = stats.shapiro(x)

    if len(x) >= 3 and x.std(ddof=1) > 0:
        z = (x - x.mean()) / x.std(ddof=1)
        inferential["kolmogorov_smirnov"] = stats.kstest(z, "norm")

    if len(x) >= 3:
        inferential["anderson"] = stats.anderson(x, dist="norm")
        inferential["jarque_bera"] = stats.jarque_bera(x)

# Group tests: t/ANOVA, variance tests and non-parametric alternatives
if primary_cat and primary_num:
    grouped = [
        (str(k), g[primary_num].values)
        for k, g in sample_df[[primary_cat, primary_num]].dropna().groupby(primary_cat)
        if len(g) >= 2
    ]
    groups = [g for _, g in grouped]

    if len(groups) == 2:
        inferential["welch_t"] = stats.ttest_ind(groups[0], groups[1], equal_var=False)
        inferential["mann_whitney"] = stats.mannwhitneyu(
            groups[0], groups[1], alternative="two-sided"
        )

        v1, v2 = np.var(groups[0], ddof=1), np.var(groups[1], ddof=1)
        if v1 > 0 and v2 > 0:
            F = v1 / v2
            dfn, dfd = len(groups[0])-1, len(groups[1])-1
            pF = 2 * min(stats.f.cdf(F, dfn, dfd), 1-stats.f.cdf(F, dfn, dfd))
            inferential["f_test_variance"] = (F, pF)

    elif len(groups) >= 3:
        inferential["anova"] = stats.f_oneway(*groups)
        inferential["kruskal_wallis"] = stats.kruskal(*groups)

    if len(groups) >= 2:
        inferential["levene"] = stats.levene(*groups, center="median")
        try:
            inferential["bartlett"] = stats.bartlett(*groups)
        except Exception:
            pass

# Proportion z-test for binary categorical field
binary_cats = [c for c in cat_cols if sample_df[c].dropna().nunique() == 2]
if binary_cats:
    bc = binary_cats[0]
    counts = sample_df[bc].fillna("Missing").astype(str).value_counts()
    success, n = int(counts.iloc[0]), int(counts.sum())
    p_hat = success / n
    z = (p_hat - .5) / math.sqrt(.25/n)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    inferential["proportion_z"] = {
        "column": bc, "category": counts.index[0],
        "count": success, "n": n, "proportion": p_hat, "z": z, "p": p
    }

# Chi-square goodness of fit and independence
if primary_cat:
    counts = sample_df[primary_cat].fillna("Missing").astype(str).value_counts()
    if len(counts) >= 2:
        inferential["chi_square_gof"] = stats.chisquare(counts.values)

low_cats = [c for c in cat_cols if 2 <= sample_df[c].nunique(dropna=True) <= 10]
if len(low_cats) >= 2:
    c1, c2 = low_cats[:2]
    ct = pd.crosstab(
        sample_df[c1].fillna("Missing"),
        sample_df[c2].fillna("Missing")
    )
    if ct.shape[0] >= 2 and ct.shape[1] >= 2:
        inferential["chi_square_independence"] = stats.chi2_contingency(ct)

print("INFERENTIAL RESULTS")
for k, v in inferential.items():
    print(k, "=>", v)

print("\nNote:")
print("Wilcoxon signed-rank and Friedman are not forced because they require paired/repeated-measures designs.")
print("If a defensible paired structure exists in the actual dataset, these tests should be run for that design.")

In [ ]:
# ============================================================
# CELL 14 — Regression / predictive analysis
# ============================================================
regression_results = {}

if len(num_cols) >= 2:
    target = amount_col if amount_col in num_cols else num_cols[0]
    predictors = [c for c in num_cols if c != target][:3]

    reg_df = sample_df[[target] + predictors].dropna()

    if len(reg_df) >= max(30, len(predictors) * 10):
        ols = sm.OLS(
            reg_df[target],
            sm.add_constant(reg_df[predictors])
        ).fit()

        regression_results["multiple_ols"] = ols
        print("MULTIPLE OLS")
        print(ols.summary())

        # Polynomial regression using the first predictor
        p = predictors[0]
        poly_df = sample_df[[target, p]].dropna()
        if len(poly_df) >= 30:
            X2 = sm.add_constant(pd.DataFrame({
                p: poly_df[p],
                "x_squared": poly_df[p] ** 2
            }))
            poly = sm.OLS(poly_df[target], X2).fit()
            regression_results["polynomial"] = poly

            print("\nDEGREE-2 POLYNOMIAL REGRESSION")
            print(poly.summary())

# Regression with categorical variables
if primary_cat and primary_num and sample_df[primary_cat].nunique(dropna=True) <= 15:
    cat_reg_df = sample_df[[primary_num, primary_cat]].dropna()
    if len(cat_reg_df) >= 30:
        try:
            categorical_model = smf.ols(
                f'Q("{primary_num}") ~ C(Q("{primary_cat}"))',
                data=cat_reg_df
            ).fit()
            regression_results["categorical"] = categorical_model
            print("\nREGRESSION WITH CATEGORICAL VARIABLE")
            print(categorical_model.summary())
        except Exception as e:
            print("Categorical regression unavailable:", e)

# Logistic regression if a binary categorical target exists
if binary_cats and num_cols:
    target_cat = binary_cats[0]
    xs = num_cols[:min(3, len(num_cols))]
    log_df = sample_df[[target_cat] + xs].dropna().copy()

    if len(log_df) >= 50:
        classes = list(log_df[target_cat].astype(str).unique())
        if len(classes) == 2:
            log_df["_y"] = (log_df[target_cat].astype(str) == classes[1]).astype(int)
            try:
                logistic = sm.Logit(
                    log_df["_y"],
                    sm.add_constant(log_df[xs])
                ).fit(disp=False)
                regression_results["logistic"] = logistic

                print("\nLOGISTIC REGRESSION")
                print(logistic.summary())

                odds_ratios = np.exp(logistic.params)
                display(pd.DataFrame({
                    "Coefficient": logistic.params,
                    "Std Error": logistic.bse,
                    "z": logistic.tvalues,
                    "p-value": logistic.pvalues,
                    "Odds Ratio": odds_ratios
                }).round(4))
            except Exception as e:
                print("Logistic regression unavailable:", e)

In [ ]:
# ============================================================
# CELL 15 — Produce concise observations and managerial insights
# ============================================================
observations = []
recommendations = []

observations.append(
    f"The fixed analytical sample contains {len(sample_df):,} records and {sample_df.shape[1]:,} variables."
)

if primary_num:
    x = sample_df[primary_num].dropna()
    observations.append(
        f"{primary_num}: mean={x.mean():,.4f}, median={x.median():,.4f}, "
        f"SD={x.std():,.4f}, min={x.min():,.4f}, max={x.max():,.4f}, "
        f"skewness={x.skew():,.4f}, kurtosis={x.kurtosis():,.4f}."
    )

if primary_cat:
    f = frequency_tables[primary_cat]
    observations.append(
        f"{primary_cat}: highest-frequency category is {f.iloc[0][primary_cat]} "
        f"with {int(f.iloc[0]['Count']):,} records "
        f"({f.iloc[0]['Relative Frequency (%)']:.2f}%)."
    )

if not correlations.empty:
    q = correlations.iloc[0]
    observations.append(
        f"Strongest absolute Pearson association: {q['X']} vs {q['Y']} "
        f"(r={q['Pearson_r']:.4f}, p={q['Pearson_p']:.4g})."
    )

recommendations.extend([
    "Use the Streamlit dashboard as the primary operational interface for interactive exploration.",
    "Monitor both mean and median when distributions are skewed and review extreme observations before classifying them as errors.",
    "Investigate material category differences with operational context rather than treating statistical significance as automatic causality.",
    "Use monthly trend monitoring where a valid date field exists.",
    "Use regression as predictive/associational evidence unless the research design supports causal inference."
])

print("OBSERVATIONS")
for x in observations:
    print("•", x)

print("\nMANAGERIAL INSIGHTS / RECOMMENDATIONS")
for x in recommendations:
    print("•", x)

In [ ]:
# ============================================================
# CELL 16 — Save the complete analysis outputs
# ============================================================
Path("outputs").mkdir(exist_ok=True)

descriptive.to_csv("outputs/descriptive_statistics.csv")
central_tendency.to_csv("outputs/central_tendency.csv")
missing_table.to_csv("outputs/missing_data_audit.csv", index=False)
variable_dictionary.to_csv("outputs/variable_dictionary.csv", index=False)
correlations.to_csv("outputs/correlation_results.csv", index=False)

for c, f in frequency_tables.items():
    safe_name = norm(c)
    f.to_csv(f"outputs/frequency_{safe_name}.csv", index=False)

print("All analysis tables have been exported to the outputs/ folder.")

In [ ]:
# ============================================================
# CELL 17 — Generate the detailed Word analytical report
# ============================================================
# The report generator is written below so the report is reproducible
# from the exact same fixed sample used by the dashboard.

report_code = Path("report_generator.py").read_text(encoding="utf-8") if Path("report_generator.py").exists() else None

if report_code is None:
    raise FileNotFoundError("report_generator.py was not found. Re-run the package extraction/setup cell.")

# Run the report generator directly from the fixed sample.
!python report_generator.py --csv data/shopease_sample_112_096_074.csv --output ShopEase_Analytical_Report.docx

print("Generated: ShopEase_Analytical_Report.docx")

In [ ]:
# ============================================================
# CELL 18 — Write / refresh the Streamlit app
# ============================================================
# The packaged app.py should be present in the same Colab working directory.
# If using the downloaded project zip, extract the package first.

required = ["app.py", "report_generator.py", "requirements.txt"]
missing = [x for x in required if not Path(x).exists()]

if missing:
    raise FileNotFoundError(f"Missing project files: {missing}")

# Ensure the Streamlit app sees the fixed sample in its expected location.
print("Streamlit app:", Path("app.py").resolve())
print("Sample CSV:", Path("data/shopease_sample_112_096_074.csv").resolve())

## 🔐 Ngrok credential setup

For a clean submission, **do not place the authtoken directly in the notebook source**.

In Google Colab:
1. Open the left sidebar → **Secrets** (key icon).
2. Add a secret named `NGROK_AUTHTOKEN`.
3. Paste the ngrok authtoken into that secret.
4. Enable notebook access for the secret.
5. Run the next cell.

If Colab Secrets are unavailable, the cell will privately prompt for the token using `getpass`.

In [ ]:
# ============================================================
# CELL 19 — Start Streamlit and create the live ngrok URL
# ============================================================
from pyngrok import ngrok

# Stop old Streamlit/ngrok processes if the notebook is re-run.
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
try:
    ngrok.kill()
except Exception:
    pass

# Read token from Colab Secret when available; otherwise prompt privately.
token = None
try:
    from google.colab import userdata
    token = userdata.get("NGROK_AUTHTOKEN")
except Exception:
    token = None

if not token:
    token = getpass.getpass("Enter your ngrok authtoken privately: ")

if not token:
    raise ValueError("No ngrok authtoken was supplied.")

ngrok.set_auth_token(token)

# Start Streamlit in the background.
log_file = open("streamlit.log", "w")
process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(8)

# Create the public tunnel.
public_url = ngrok.connect(8501).public_url

print("=" * 70)
print("LIVE STREAMLIT DASHBOARD")
print(public_url)
print("=" * 70)
print("Keep this Colab runtime alive while demonstrating the dashboard.")

## ✅ Final submission checklist

- [ ] Project title and student details verified.
- [ ] Correct group ID and seed verified.
- [ ] Exactly 2,500 fixed random records used.
- [ ] Kaggle source link included.
- [ ] Variable dictionary completed.
- [ ] Missing-data audit completed.
- [ ] Numeric descriptive statistics completed.
- [ ] Categorical frequency / relative frequency completed.
- [ ] Pearson and Spearman correlation completed where applicable.
- [ ] Required plots completed where the data structure supports them.
- [ ] Confidence interval completed.
- [ ] Mean, variance and non-parametric tests completed where applicable.
- [ ] Normality tests completed.
- [ ] Chi-square tests completed where applicable.
- [ ] Regression models completed where the schema supports them.
- [ ] Word report generated from the exact fixed sample.
- [ ] Streamlit dashboard tested.
- [ ] Live ngrok dashboard link copied.
- [ ] Colab share link copied.
- [ ] Appendix 1 contribution details updated for all members.
- [ ] Appendix 2 replaced with the group's actual AI prompt log if required.